# Modelos secuenciales en Keras: RNN, LSTM y GRU con IMDB

## Objetivo

En este notebook trabajaremos con datos secuenciales usando Keras.

En particular, veremos cómo:

- cargar un dataset de texto,
- preparar secuencias de entrada,
- construir un modelo con `Embedding + SimpleRNN`,
- comparar luego con `LSTM` y `GRU`,
- y evaluar cuál funciona mejor en una tarea simple de clasificación de texto.

La idea no es solo entrenar modelos, sino también entender por qué las arquitecturas recurrentes son adecuadas para secuencias.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

## Hoja de ruta

Trabajaremos en los siguientes pasos:

1. Cargar el dataset IMDB.
2. Entender cómo vienen representadas las reseñas.
3. Preparar secuencias con padding.
4. Construir un primer modelo con `Embedding + SimpleRNN`.
5. Entrenarlo y evaluarlo.
6. Repetir la idea con `LSTM` y `GRU`.
7. Comparar resultados.

## 1. Cargar el dataset IMDB

Usaremos el dataset **IMDB** incluido en Keras.

Este dataset contiene reseñas de películas etiquetadas como:

- `0`: reseña negativa
- `1`: reseña positiva

A diferencia de lo que hicimos antes en clasificación binaria con MLP, aquí trabajaremos con las reseñas como **secuencias de palabras codificadas por enteros**.

In [ ]:
from tensorflow.keras.datasets import imdb

max_features = 10000  # tamaño del vocabulario

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

print("Número de ejemplos de entrenamiento:", len(X_train))
print("Número de ejemplos de test:", len(X_test))

In [ ]:
print("Primera reseña (codificada como enteros):")
print(X_train[0][:40])

print("\nEtiqueta de la primera reseña:")
print(y_train[0])

### Observación

Cada reseña viene como una **lista de enteros**.

Cada entero representa una palabra del vocabulario.  
Por lo tanto, cada reseña es una **secuencia** de longitud variable.

Eso hace que este dataset sea muy apropiado para usar capas como:

- `Embedding`
- `SimpleRNN`
- `LSTM`
- `GRU`

In [ ]:
lengths = [len(seq) for seq in X_train]

print("Longitud mínima:", np.min(lengths))
print("Longitud máxima:", np.max(lengths))
print("Longitud promedio:", np.mean(lengths))

In [ ]:
plt.figure(figsize=(7,4))
plt.hist(lengths, bins=50)
plt.xlabel("Longitud de la reseña")
plt.ylabel("Frecuencia")
plt.title("Distribución de longitudes en IMDB")
plt.show()

## 2. Padding de secuencias

Como las reseñas tienen distinta longitud, necesitamos transformarlas a una longitud común para poder entrenar en batches.

Usaremos `pad_sequences`, que:
- recorta secuencias largas,
- y rellena secuencias más cortas con ceros.

Esto no cambia la idea central del problema: seguimos trabajando con secuencias ordenadas.

In [ ]:
from tensorflow.keras.utils import pad_sequences

maxlen = 200

X_train_seq = pad_sequences(X_train, maxlen=maxlen)
X_test_seq  = pad_sequences(X_test, maxlen=maxlen)

print("Shape de X_train_seq:", X_train_seq.shape)
print("Shape de X_test_seq:", X_test_seq.shape)

In [ ]:
print("Primera reseña después de padding:")
print(X_train_seq[0])

Por defecto, `pad_sequences(..., maxlen=200)`:
- rellena al inicio (`padding="pre"`),
- y si la secuencia es más larga que 200, recorta desde el inicio (`truncating="pre"`).

Es decir, conserva los últimos 200 tokens de cada reseña.

### Comentario

Ahora cada ejemplo tiene shape `(200,)`.

Eso significa que cada reseña se representa como una secuencia de 200 posiciones.

El siguiente paso será transformar esos enteros en vectores densos mediante una capa `Embedding`.

## 3. Primer modelo: `Embedding + SimpleRNN`

La estructura más simple que veremos será:

- una capa `Embedding`, que convierte cada palabra en un vector,
- una capa `SimpleRNN`, que procesa la secuencia,
- y una capa `Dense` final para clasificación binaria.

Este modelo nos sirve como punto de partida conceptual.

In [ ]:
simple_rnn_model = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(input_dim=max_features, output_dim=32),
    layers.SimpleRNN(32),
    layers.Dense(1, activation="sigmoid")
])

simple_rnn_model.summary()

### Nota sobre la arquitectura

- `Embedding(input_dim=max_features, output_dim=32)`:
  cada palabra se convierte en un vector de dimensión 32.

- `SimpleRNN(32)`:
  procesa la secuencia completa y produce una representación final.

- `Dense(1, activation="sigmoid")`:
  entrega una probabilidad para clasificación binaria.

In [9]:
simple_rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

## 4. Entrenamiento del modelo `SimpleRNN`

Usaremos una fracción del conjunto de entrenamiento como validación para monitorear el desempeño durante el entrenamiento.

In [ ]:
history_simple_rnn = simple_rnn_model.fit(
    X_train_seq,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

In [ ]:
simple_test_loss, simple_test_acc = simple_rnn_model.evaluate(X_test_seq, y_test, verbose=0)

print("Test loss (SimpleRNN):", round(simple_test_loss, 4))
print("Test accuracy (SimpleRNN):", round(simple_test_acc, 4))

## 5. Curvas de entrenamiento

Como en sesiones anteriores, es útil observar cómo evolucionan:

- la pérdida,
- y la accuracy,

tanto en entrenamiento como en validación.

In [12]:
def plot_history(history, title_prefix="Modelo"):
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]
    loss = history.history["loss"]
    val_loss = history.history["val_loss"]
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(7,4))
    plt.plot(epochs, acc, label="train accuracy")
    plt.plot(epochs, val_acc, label="val accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title_prefix}: accuracy")
    plt.legend()
    plt.show()

    plt.figure(figsize=(7,4))
    plt.plot(epochs, loss, label="train loss")
    plt.plot(epochs, val_loss, label="val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix}: loss")
    plt.legend()
    plt.show()

In [ ]:
plot_history(history_simple_rnn, title_prefix="SimpleRNN")

### Comentario

`SimpleRNN` nos permite modelar secuencias, pero en problemas reales puede tener dificultades para capturar dependencias más largas.

Por eso ahora compararemos con dos variantes más robustas:

- `LSTM`
- `GRU`

## 6. Segundo modelo: `Embedding + LSTM`

La capa `LSTM` fue diseñada para manejar mejor dependencias de más largo alcance.

Mantendremos la misma lógica general del modelo anterior para que la comparación sea más justa.

In [ ]:
lstm_model = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(input_dim=max_features, output_dim=32),
    layers.LSTM(32),
    layers.Dense(1, activation="sigmoid")
])

lstm_model.summary()

In [15]:
lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_lstm = lstm_model.fit(
    X_train_seq,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

In [ ]:
lstm_test_loss, lstm_test_acc = lstm_model.evaluate(X_test_seq, y_test, verbose=0)

print("Test loss (LSTM):", round(lstm_test_loss, 4))
print("Test accuracy (LSTM):", round(lstm_test_acc, 4))

In [ ]:
plot_history(history_lstm, title_prefix="LSTM")

## 7. Tercer modelo: `Embedding + GRU`

La capa `GRU` es una alternativa a LSTM con una estructura más simple, pero muy efectiva en muchos problemas prácticos.

In [ ]:
gru_model = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(input_dim=max_features, output_dim=32),
    layers.GRU(32),
    layers.Dense(1, activation="sigmoid")
])

gru_model.summary()

In [20]:
gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_gru = gru_model.fit(
    X_train_seq,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

In [ ]:
gru_test_loss, gru_test_acc = gru_model.evaluate(X_test_seq, y_test, verbose=0)

print("Test loss (GRU):", round(gru_test_loss, 4))
print("Test accuracy (GRU):", round(gru_test_acc, 4))

In [ ]:
plot_history(history_gru, title_prefix="GRU")

## 8. Una variante práctica: dropout recurrente

En sesiones anteriores vimos que `dropout` es una técnica útil para reducir sobreajuste.

En modelos recurrentes aparece además la opción `recurrent_dropout`, que aplica regularización sobre las conexiones recurrentes internas de la capa.

No siempre es necesario, pero es útil conocerlo porque aparece con frecuencia en la práctica.

In [ ]:
gru_dropout_model = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(input_dim=max_features, output_dim=32),
    layers.GRU(32, dropout=0.2, recurrent_dropout=0.2),
    layers.Dense(1, activation="sigmoid")
])

gru_dropout_model.summary()

In [25]:
gru_dropout_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_gru_dropout = gru_dropout_model.fit(
    X_train_seq,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

In [ ]:
gru_dropout_test_loss, gru_dropout_test_acc = gru_dropout_model.evaluate(X_test_seq, y_test, verbose=0)

print("Test loss (GRU con dropout recurrente):", round(gru_dropout_test_loss, 4))
print("Test accuracy (GRU con dropout recurrente):", round(gru_dropout_test_acc, 4))

In [ ]:
plot_history(history_gru_dropout, title_prefix="GRU con dropout recurrente")

### Comentario

Aquí usamos dos tipos de regularización:

- `dropout=0.2`: actúa sobre la entrada de la capa,
- `recurrent_dropout=0.2`: actúa sobre las conexiones recurrentes internas.

La idea es parecida a la que vimos antes con MLP: ayudar a la generalización.  
En algunos casos mejora el desempeño; en otros, simplemente vuelve el entrenamiento más estable.

## 9. Comparación final

In [ ]:
model_names = ["SimpleRNN", "LSTM", "GRU", "GRU + rec. dropout"]
test_accuracies = [simple_test_acc, lstm_test_acc, gru_test_acc, gru_dropout_test_acc]

for name, acc in zip(model_names, test_accuracies):
    print(f"{name}: test accuracy = {acc:.4f}")

In [ ]:
plt.figure(figsize=(7,4))
plt.bar(model_names, test_accuracies)
plt.ylim(0.7, 0.95)
plt.ylabel("Test accuracy")
plt.title("Comparación de arquitecturas recurrentes")
plt.xticks(rotation=15)
plt.show()

### Interpretación

Aunque el resultado exacto puede variar entre ejecuciones, en muchos casos veremos que:

- `SimpleRNN` sirve como punto de partida conceptual,
- `LSTM` suele capturar mejor dependencias más largas,
- `GRU` suele ofrecer un desempeño competitivo con una estructura algo más simple.

## 10. Algunas predicciones de ejemplo

In [ ]:
y_prob_gru = gru_model.predict(X_test_seq[:10], verbose=0)
y_pred_gru = (y_prob_gru > 0.5).astype("int32").ravel()

for i in range(10):
    print(f"Ejemplo {i+1}:")
    print("  Probabilidad positiva:", round(float(y_prob_gru[i]), 4))
    print("  Predicción:", y_pred_gru[i])
    print("  Etiqueta real:", y_test[i])
    print()

## Bonuses opcionales

A continuación veremos algunas extensiones de los modelos recurrentes básicos.

Estas variantes **no son necesarias** para entender la idea principal de la sesión, pero sirven para mostrar cómo Keras permite construir modelos secuenciales más ricos.

Veremos tres ejemplos:

- **stacking** de capas recurrentes,
- **Bidirectional LSTM**,
- y un enfoque alternativo con **Conv1D** para secuencias.

## Bonus opcional: stacking de capas recurrentes

También es posible construir modelos recurrentes con más de una capa.

A esta idea a veces se le llama **stacking** de capas recurrentes.

La idea general es:

- una primera capa recurrente procesa la secuencia,
- su salida completa alimenta a una segunda capa recurrente,
- y luego una capa final produce la clasificación.

### Punto importante en Keras

Si una capa recurrente será seguida por otra capa recurrente, entonces la primera debe usar:

`return_sequences=True`

¿Por qué?

Porque la segunda capa recurrente necesita recibir una **secuencia completa**, no solo un único vector final.

In [ ]:
stacked_lstm_model = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(input_dim=max_features, output_dim=32),
    layers.LSTM(32, return_sequences=True),
    layers.LSTM(32),
    layers.Dense(1, activation="sigmoid")
])

stacked_lstm_model.summary()

### Qué está pasando aquí

- La primera capa `LSTM(32, return_sequences=True)` entrega una salida para **cada paso temporal**.
- La segunda capa `LSTM(32)` recibe esa secuencia y la resume en una representación final.
- Luego la capa `Dense` produce la salida binaria.

En otras palabras:

- **sin** `return_sequences=True`, la primera LSTM entregaría solo un vector final;
- **con** `return_sequences=True`, entrega la secuencia completa de estados ocultos.

In [34]:
stacked_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_stacked_lstm = stacked_lstm_model.fit(
    X_train_seq,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

In [ ]:
stacked_lstm_test_loss, stacked_lstm_test_acc = stacked_lstm_model.evaluate(
    X_test_seq, y_test, verbose=0
)

print("Test loss (Stacked LSTM):", round(stacked_lstm_test_loss, 4))
print("Test accuracy (Stacked LSTM):", round(stacked_lstm_test_acc, 4))

In [ ]:
plot_history(history_stacked_lstm, title_prefix="Stacked LSTM")

### Comentario sobre el bonus

Apilar capas recurrentes puede permitir que el modelo aprenda representaciones más ricas.

Sin embargo:

- aumenta el número de parámetros,
- sube el costo computacional,
- y no siempre garantiza una mejora clara.

Para este curso, lo importante es entender la lógica:

si una capa recurrente alimenta a otra, la capa intermedia debe devolver la secuencia completa usando `return_sequences=True`.

## Bonus opcional: Bidirectional LSTM

Hasta ahora hemos usado modelos recurrentes que recorren la secuencia en una sola dirección.

Una idea interesante es procesar la secuencia en ambos sentidos:

- de izquierda a derecha,
- y de derecha a izquierda.

En Keras, esto se puede hacer envolviendo una capa recurrente con `Bidirectional`.

No es necesario para entender la idea principal de la sesión, pero sirve para ver una extensión muy usada en la práctica.

In [ ]:
bidirectional_lstm_model = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(input_dim=max_features, output_dim=32),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(1, activation="sigmoid")
])

bidirectional_lstm_model.summary()

In [39]:
bidirectional_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_bilstm = bidirectional_lstm_model.fit(
    X_train_seq,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

In [ ]:
bilstm_test_loss, bilstm_test_acc = bidirectional_lstm_model.evaluate(X_test_seq, y_test, verbose=0)

print("Test loss (Bidirectional LSTM):", round(bilstm_test_loss, 4))
print("Test accuracy (Bidirectional LSTM):", round(bilstm_test_acc, 4))

In [ ]:
plot_history(history_bilstm, title_prefix="Bidirectional LSTM")

### Comentario sobre el bonus

Un modelo bidireccional puede aprovechar mejor el contexto de la secuencia completa, porque procesa la información en ambos sentidos.

En algunos problemas esto puede mejorar el desempeño. Sin embargo, también agrega complejidad y costo computacional.

Para este curso, lo importante es reconocer que existen extensiones de las RNN/LSTM/GRU básicas que permiten construir modelos más potentes.

In [43]:
model_names_bonus = ["SimpleRNN", "LSTM", "GRU", "BiLSTM"]
test_accuracies_bonus = [simple_test_acc, lstm_test_acc, gru_test_acc, bilstm_test_acc]

for name, acc in zip(model_names_bonus, test_accuracies_bonus):
    print(f"{name}: test accuracy = {acc:.4f}")

SimpleRNN: test accuracy = 0.7955
LSTM: test accuracy = 0.8611
GRU: test accuracy = 0.8554
BiLSTM: test accuracy = 0.8605


In [ ]:
plt.figure(figsize=(7,4))
plt.bar(model_names_bonus, test_accuracies_bonus)
plt.ylim(0.7, 0.95)
plt.ylabel("Test accuracy")
plt.title("Comparación incluyendo bonus")
plt.show()

## Bonus opcional 2: un enfoque con `Conv1D`

Aunque hoy nos centramos en modelos recurrentes, también es posible trabajar con secuencias usando convoluciones 1D.

La idea es que `Conv1D` puede detectar patrones locales en la secuencia, de forma análoga a como una CNN detecta patrones locales en imágenes.

No reemplaza conceptualmente a RNN/LSTM/GRU, pero es una alternativa muy usada en problemas de texto y señales.

In [ ]:
conv1d_model = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(input_dim=max_features, output_dim=32),
    layers.Conv1D(filters=32, kernel_size=3, activation="relu"),
    layers.GlobalMaxPooling1D(),
    layers.Dense(1, activation="sigmoid")
])

conv1d_model.summary()

In [46]:
conv1d_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_conv1d = conv1d_model.fit(
    X_train_seq,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

In [ ]:
conv1d_test_loss, conv1d_test_acc = conv1d_model.evaluate(X_test_seq, y_test, verbose=0)

print("Test loss (Conv1D):", round(conv1d_test_loss, 4))
print("Test accuracy (Conv1D):", round(conv1d_test_acc, 4))

In [ ]:
plot_history(history_conv1d, title_prefix="Conv1D")

### Comentario sobre `Conv1D`

Este modelo ya no usa recurrencia, pero sigue siendo adecuado para secuencias.

La convolución 1D busca patrones locales en ventanas pequeñas de la secuencia, mientras que el `GlobalMaxPooling1D` resume la activación más importante encontrada.

Es una alternativa interesante porque suele ser más liviana y rápida de entrenar.

In [ ]:
model_names_bonus = [
    "SimpleRNN",
    "LSTM",
    "GRU",
    "GRU + rec. dropout",
    "Stacked LSTM",
    "BiLSTM",
    "Conv1D"
]

test_accuracies_bonus = [
    simple_test_acc,
    lstm_test_acc,
    gru_test_acc,
    gru_dropout_test_acc,
    stacked_lstm_test_acc,
    bilstm_test_acc,
    conv1d_test_acc
]

for name, acc in zip(model_names_bonus, test_accuracies_bonus):
    print(f"{name}: test accuracy = {acc:.4f}")

In [ ]:
plt.figure(figsize=(9,4))
plt.bar(model_names_bonus, test_accuracies_bonus)
plt.ylim(0.7, 0.95)
plt.ylabel("Test accuracy")
plt.title("Comparación incluyendo variantes y bonuses")
plt.xticks(rotation=20)
plt.show()

### Combinando `CNNs` y `RNNs` para procesar secuencias largas

* Una estrategia para combinar la velocidad y la ligereza de las CNNs con la sensibilidad del orden de las RNNs es utilizar una CNN 1D como un paso de preprocesamiento antes de una RNNs, tal como en la siguiente figura.

![](https://drive.google.com/uc?id=1KGPD0KrsZyJ9IdIewoTT_NAfJGOLM5_v)


* Esto es especialmente beneficioso cuando se trata de secuencias que son tan largas que no pueden procesarse de manera realista con una RNN, como secuencias con miles de pasos.

* La convnet convertirá la secuencia de entrada larga en secuencias mucho más cortas (muestras reducidas) de características de nivel superior.

* Esta secuencia de características extraídas se convierte en la entrada a la parte del RNN de la red.

En el siguiente modelo, se comienza con dos capas `Conv1D` y siguiendo con una capa `GRU`.

In [ ]:
conv1d_gru_model = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(input_dim=max_features, output_dim=32),
    layers.Conv1D(filters=32, kernel_size=3, activation="relu"),
    layers.MaxPooling1D(2),
    layers.Conv1D(filters=32, kernel_size=3, activation="relu"),
    layers.GRU(32),
    layers.Dense(1, activation="sigmoid")
])

conv1d_gru_model.summary()

In [53]:
conv1d_gru_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history_conv1d_gru = conv1d_gru_model.fit(
    X_train_seq,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)

In [ ]:
conv1d_gru_test_loss, conv1d_gru_test_acc = conv1d_gru_model.evaluate(X_test_seq, y_test, verbose=0)

print("Test loss (Conv1D+GRU):", round(conv1d_gru_test_loss, 4))
print("Test accuracy (Conv1D+GRU):", round(conv1d_gru_test_acc, 4))

In [ ]:
plot_history(history_conv1d_gru, title_prefix="Conv1D+gru")

## Cierre final

En este notebook vimos varias formas de trabajar con secuencias de texto en Keras:

- `Embedding + SimpleRNN`
- `Embedding + LSTM`
- `Embedding + GRU`
- `Embedding + GRU` con `recurrent_dropout`

y, como extensiones opcionales:

- `Embedding + LSTM con stacking`
- `Embedding + Bidirectional(LSTM)`
- `Embedding + Conv1D`
- `Embedding + Conv1D + GRU`

La idea principal es que, cuando el orden importa, necesitamos arquitecturas capaces de explotar la estructura secuencial de los datos.

En el siguiente notebook veremos una aplicación complementaria a otro tipo de dato secuencial.